# Glance Fashion Retrieval - Notebook 1: Data Pipeline & Indexer
Run on Kaggle with **T4 GPU** enabled.
Attach the `fashionpedia-dataset` and `glance-fashion-retrieval-code` datasets before running.

In [ ]:
import os, shutil, sys, warnings
warnings.filterwarnings('ignore')

# 1. Find and copy codebase
code_source = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'src' in dirs and 'config.py' in os.listdir(os.path.join(root, 'src')):
        code_source = root
        break

if not code_source:
    raise FileNotFoundError('Could not find the codebase! Did you attach the code dataset?')

work_path = '/kaggle/working/glance-code'
if not os.path.exists(work_path):
    shutil.copytree(code_source, work_path)
    print('Copied code to: ' + work_path)
else:
    print('Code already exists at: ' + work_path)

if work_path not in sys.path:
    sys.path.insert(0, work_path)


print('Setup complete!')

In [ ]:
# Install dependencies AFTER copying so requirements.txt is available
!pip install -q -r /kaggle/working/glance-code/requirements.txt
!pip install -q git+https://github.com/huggingface/transformers.git

## Step 1 - Subsample 3,000 images

In [ ]:
from src.data_pipeline.subsample import subsample_images
print('Subsampling images...')
_ = subsample_images()

## Step 2 - Parse COCO Annotations into Tags

In [ ]:
from src.data_pipeline.parse_annotations import parse_annotations
print('Parsing annotations...')
_ = parse_annotations()

## Step 3 - Generate Captions (Rule-Based, No API Key Required)
> Runs entirely on CPU using structured Fashionpedia tags. Completes in ~2 minutes with no external calls.

In [ ]:
from src.data_pipeline.generate_captions import generate_all_captions
print('Generating captions via Gemini...')
generate_all_captions(resume=True)

## Step 4 - Build ChromaDB Vector Index

In [ ]:
from src.indexer import build_index
print('Building vector index with Marqo-FashionSigLIP...')
build_index()